<a href="https://colab.research.google.com/github/deendiran/plp-da-a1-gikomba-stall-analysis/blob/main/stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gikomba Market Fresh-Produce Stall Analysis

This notebook performs a comprehensive performance and inventory analysis for Wanjiku's fresh-produce stall at Gikomba Market using core Python only (no external libraries such as Pandas or NumPy).


In [4]:
# Data cell exactly as provided
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,

               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2



# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg

stock = {

    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},

    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},

    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},

    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},

    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},

}

## Task 1: Overall Sales Metrics and Best/Worst Days

We calculate basic descriptive metrics (sum, average, max, and min) for the 14-day sales period using Python's built-in sum(), len(), max(), and min() functions. We then use list.index() to identify the positions of the highest and lowest sales days. Since each week consists of 7 days, integer division (index // 7) + 1 determines whether the day belongs to Week 1 or Week 2.

In [5]:
# 1. Overall metrics
total_sales = sum(daily_sales)
total_days = len(daily_sales)
average_sales = total_sales / total_days
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

# 2. Locate best and worst days
best_index = daily_sales.index(highest_sales)
worst_index = daily_sales.index(lowest_sales)

best_day_name = days[best_index]
best_week_num = (best_index // 7) + 1

worst_day_name = days[worst_index]
worst_week_num = (worst_index // 7) + 1

# Output the results
print(f"Total Sales (14 Days): KES {total_sales:,.2f}")
print(f"Average Daily Sales:  KES {average_sales:,.2f}")
print(f"Highest Sales Day:    KES {highest_sales:,.2f} ({best_day_name}, Week {best_week_num})")
print(f"Lowest Sales Day:     KES {lowest_sales:,.2f} ({worst_day_name}, Week {worst_week_num})")

Total Sales (14 Days): KES 46,450.00
Average Daily Sales:  KES 3,317.86
Highest Sales Day:    KES 5,600.00 (Sat, Week 2)
Lowest Sales Day:     KES 1,950.00 (Tue, Week 1)


## Task 2: Weekly Totals and Week-on-Week Growth

We define a reusable function week_totals(sales) that partitions the sales data into two 7-day segments via list slicing ([:7] and [7:]) and returns their sums as a tuple (week 1 total, week 2 total). We then unpack the tuple and compute the percentage growth from Week 1 to Week 2.

In [6]:
def week_totals(sales):
  """
    Computes total sales for Week 1 and Week 2 using list slicing.
    Parameters:
        sales (list): A list of daily sales figures covering 14 days.
    Returns:
        tuple: (week_1_total, week_2_total) containing sales sums for each week.
  """

  week_1 = sum(sales[:7])
  week_2 = sum(sales[7:])

  return (week_1, week_2)

# Unpack the returned tuple
week_1_total, week_2_total = week_totals(daily_sales)

# Unpack the returned tuple
week_1_total, week_2_total = week_totals(daily_sales)
# Calculate week-on-week growth percentage
wow_growth_pct = ((week_2_total - week_1_total) / week_1_total) * 100
print(f"Week 1 Total: KES {week_1_total:,.2f}")
print(f"Week 2 Total: KES {week_2_total:,.2f}")
print(f"Week-on-Week Growth: {wow_growth_pct:.2f}%")

Week 1 Total: KES 22,050.00
Week 2 Total: KES 24,400.00
Week-on-Week Growth: 10.66%


## Task 3: Daily Sales Classification and Distribution

We define classify_day(amount, average) to categorize daily sales as "Strong" (> 20% above average), "Weak" (> 20% below average), or "Normal". We iterate through all 14 days and track occurrences using a dictionary counting accumulator pattern.

In [7]:
def classify_day(amount, average):
    """
    Classifies a single day's sales performance relative to the average.

    Parameters:
        amount (float or int): Sales amount for the day.
        average (float or int): Overall average daily sales.

    Returns:
        str: 'Strong' (>20% above avg), 'Weak' (>20% below avg), or 'Normal'.
    """
    if amount > average * 1.20:
        return "Strong"
    elif amount < average * 0.80:
        return "Weak"
    else:
        return "Normal"


# Initialize count dictionary
day_counts = {"Strong": 0, "Normal": 0, "Weak": 0}

# Classify each day and update counts
for sale in daily_sales:
    category = classify_day(sale, average_sales)
    day_counts[category] = day_counts.get(category, 0) + 1

# Print distribution
for category, count in day_counts.items():
    print(f"{category} Days: {count}")

Strong Days: 4
Normal Days: 4
Weak Days: 6


## Task 4: Weekdays vs. Weekends Comparison

Because the 7-day calendar cycles twice, indices where index % 7 < 5 correspond to weekdays (Monday–Friday), while indices where index % 7 >= 5 represent weekends (Saturday–Sunday). We filter the daily sales using list comprehensions, compute both averages, and find the percentage premium of weekend sales relative to weekday sales.

In [8]:
# Separate weekday and weekend sales
weekday_sales = [daily_sales[i] for i in range(len(daily_sales)) if i % 7 < 5]
weekend_sales = [daily_sales[i] for i in range(len(daily_sales)) if i % 7 >= 5]

weekday_average = sum(weekday_sales) / len(weekday_sales)
weekend_average = sum(weekend_sales) / len(weekend_sales)

# Difference as a percentage relative to weekdays
weekend_diff_pct = ((weekend_average - weekday_average) / weekday_average) * 100

print(f"Weekday Average (10 days): KES {weekday_average:,.2f}")
print(f"Weekend Average (4 days):  KES {weekend_average:,.2f}")
print(f"Weekend Sales Premium:     {weekend_diff_pct:.2f}% higher than weekdays")

Weekday Average (10 days): KES 2,685.00
Weekend Average (4 days):  KES 4,900.00
Weekend Sales Premium:     82.50% higher than weekdays


## Task 5: Stock Reorder Assessment and Profit Margins

We create a function `check_reorder(stock_data)` that iterates through the inventory dictionary using `.items()` to identify items with quantity at or below their reorder threshold (`qty <= reorder`). In addition, we compute each item's gross profit margin percentage using `(price - cost) / price * 100`.

In [9]:
def check_reorder(stock_data):
    """
    Evaluates stock levels and returns products needing replenishment.

    Parameters:
        stock_data (dict): Dictionary of products with 'qty' and 'reorder' thresholds.

    Returns:
        list: Names of products where current quantity is at or below the reorder level.
    """
    needs_reorder = []
    for product, details in stock_data.items():
        if details["qty"] <= details["reorder"]:
            needs_reorder.append(product)
    return needs_reorder


# Identify products that need restocking
reorder_needed = check_reorder(stock)

# Calculate gross margins for all products
margins = {}
for product, details in stock.items():
    margin_pct = ((details["price"] - details["cost"]) / details["price"]) * 100
    margins[product] = margin_pct

print(f"Products requiring reorder: {reorder_needed}")
print("\nGross Profit Margins:")
for product, margin in margins.items():
    print(f" - {product:<10}: {margin:.2f}%")

Products requiring reorder: ['Tomatoes', 'Sukuma']

Gross Profit Margins:
 - Tomatoes  : 33.33%
 - Onions    : 30.00%
 - Sukuma    : 40.00%
 - Potatoes  : 33.33%
 - Avocado   : 42.86%


## Task 6: Stall Performance Report
  
We assemble all key findings into a clean, aligned, and professional text report using formatted Python f-strings. All currency figures are displayed as `KES` with thousands separators.

In [10]:
report_header = "WANJIKU'S GIKOMBA STALL - 14-DAY PERFORMANCE REPORT"
divider_major = "=" * 60
divider_minor = "-" * 60

print(divider_major)
print(f"{report_header:^60}")
print(divider_major)

print("\n1. OVERALL SALES SUMMARY")
print(divider_minor)
print(f"  Total Revenue (14 Days)  : KES {total_sales:,.2f}")
print(f"  Daily Average Sales      : KES {average_sales:,.2f}")
print(f"  Best Day                 : KES {highest_sales:,.2f} ({best_day_name}, Week {best_week_num})")
print(f"  Worst Day                : KES {lowest_sales:,.2f} ({worst_day_name}, Week {worst_week_num})")

print("\n2. WEEKLY GROWTH & DAILY CLASSIFICATION")
print(divider_minor)
print(f"  Week 1 Total             : KES {week_1_total:,.2f}")
print(f"  Week 2 Total             : KES {week_2_total:,.2f}")
print(f"  Week-on-Week Growth      : {wow_growth_pct:+.2f}%")
print(f"  Daily Distribution       : {day_counts['Strong']} Strong | {day_counts['Normal']} Normal | {day_counts['Weak']} Weak")

print("\n3. TRAFFIC PATTERNS (WEEKDAY VS. WEEKEND)")
print(divider_minor)
print(f"  Weekday Average (Mon-Fri): KES {weekday_average:,.2f}")
print(f"  Weekend Average (Sat-Sun): KES {weekend_average:,.2f}")
print(f"  Weekend Sales Surge      : +{weekend_diff_pct:.2f}% over weekdays")

print("\n4. INVENTORY & PROFIT MARGIN STATUS")
print(divider_minor)
print(f"  {'Product':<12} {'Stock (kg)':<12} {'Reorder (kg)':<14} {'Status':<10} {'Margin':<8}")
print(divider_minor)
for product, details in stock.items():
    status = "REORDER" if details["qty"] <= details["reorder"] else "OK"
    margin = margins[product]
    print(f"  {product:<12} {details['qty']:<12} {details['reorder']:<14} {status:<10} {margin:>5.1f}%")

print(divider_major)
print(f"  Restock Alert: {', '.join(reorder_needed)} are at or below reorder level.")
print(divider_major)

    WANJIKU'S GIKOMBA STALL - 14-DAY PERFORMANCE REPORT     

1. OVERALL SALES SUMMARY
------------------------------------------------------------
  Total Revenue (14 Days)  : KES 46,450.00
  Daily Average Sales      : KES 3,317.86
  Best Day                 : KES 5,600.00 (Sat, Week 2)
  Worst Day                : KES 1,950.00 (Tue, Week 1)

2. WEEKLY GROWTH & DAILY CLASSIFICATION
------------------------------------------------------------
  Week 1 Total             : KES 22,050.00
  Week 2 Total             : KES 24,400.00
  Week-on-Week Growth      : +10.66%
  Daily Distribution       : 4 Strong | 4 Normal | 6 Weak

3. TRAFFIC PATTERNS (WEEKDAY VS. WEEKEND)
------------------------------------------------------------
  Weekday Average (Mon-Fri): KES 2,685.00
  Weekend Average (Sat-Sun): KES 4,900.00
  Weekend Sales Surge      : +82.50% over weekdays

4. INVENTORY & PROFIT MARGIN STATUS
------------------------------------------------------------
  Product      Stock (kg)   Reorder

## Recommendations for Wanjiku

1. **Restock High-Yield Produce Immediately:** Wanjiku should immediately replenish **Tomatoes** (currently 18 kg vs. 25 kg reorder point) and **Sukuma** (9 kg vs. 15 kg reorder point), as Sukuma yields an attractive **40.00% gross margin** and Tomatoes form essential daily kitchen baskets.
2. **Capitalize on the Weekend Traffic Surge:** Weekend daily revenue averages KES 4,900.00, representing an 82.50% surge over weekday sales (KES 2,685.00); thus Wanjiku should schedule with suppliers for bulk deliveries on Thursday or Friday to avoid stockouts on Saturdays and Sundays.
3. **Promote High-Margin Products:** Avocados carry the highest margin in the stall at **42.86%** (KES 60 profit per kg); pairing avocados in bundled combo deals with Sukuma, Tomatoes or Potatoes (sukuma has a margin of 40% tomatoes and both potatoes have a margin of 33.3%) will boost basket value without diluting profitability.
4. **Smooth Midweek Sales Dips:** With 6 out of 14 days classified as "Weak" and Tuesday averaging as low as **KES 1,950.00**, Wanjiku should introduce mini-promotions or wholesale bundles for local eateries or offers for their loyal customers on Tuesdays and Thursdays to lift midweek revenue.
